In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
from scipy.stats import ks_2samp
from numpy.random import default_rng

# ----------  Energy calculation (Mousley et al.)  ---------- #

def gnm_energy(A_obs, A_sim, dist_mat):
    """
    Compute energy = max(KS_degree, KS_clustering, KS_betweenness, KS_edge_length).
    A_obs and A_sim must be binary and undirected.
    """
    def prep(A):
        A_bin = (A > 0).astype(int)
        A_bin = np.triu(A_bin, k=1)
        return A_bin + A_bin.T

    # binarize, ensure symmetry & zero diagonal
    B_obs = prep(A_obs)
    B_sim = prep(A_sim)

    G_obs = nx.from_numpy_array(B_obs)
    G_sim = nx.from_numpy_array(B_sim)

    # Degree
    deg_obs = np.array([d for _, d in G_obs.degree()])
    deg_sim = np.array([d for _, d in G_sim.degree()])
    ks_degree = ks_2samp(deg_obs, deg_sim).statistic

    # Clustering coefficient
    cc_obs = np.array(list(nx.clustering(G_obs).values()))
    cc_sim = np.array(list(nx.clustering(G_sim).values()))
    ks_cc = ks_2samp(cc_obs, cc_sim).statistic

    # Betweenness centrality
    bc_obs = np.array(list(nx.betweenness_centrality(G_obs, normalized=True).values()))
    bc_sim = np.array(list(nx.betweenness_centrality(G_sim, normalized=True).values()))
    ks_bc = ks_2samp(bc_obs, bc_sim).statistic

    # Edge length distributions (using Euclidean distance matrix)
    def edge_lengths(G):
        return np.array([dist_mat[i, j] for i, j in G.edges()])
    el_obs = edge_lengths(G_obs)
    el_sim = edge_lengths(G_sim)
    ks_el = ks_2samp(el_obs, el_sim).statistic

    energy = max(ks_degree, ks_cc, ks_bc, ks_el)
    return energy


# ----------  GNM simulation  ---------- #

def compute_homophily_matrix(A):
    A = (A > 0).astype(int)
    shared = A @ A
    deg = A.sum(axis=1)
    denom = deg[:, None] + deg[None, :] - shared
    with np.errstate(divide='ignore', invalid='ignore'):
        k = np.where(denom > 0, shared / denom, 0.0)
    np.fill_diagonal(k, 0.0)
    return k

def generate_gnm(A_seed, target_m, dist_mat, gamma, eta, rng=None, eps=1e-12):
    """
    Grow until A has target_m undirected edges.
    P(i,j) ∝ (d_ij)**eta * (k_ij)**gamma, with eps-stabilization.
    """
    rng = np.random.default_rng() if rng is None else rng
    A = A_seed.copy().astype(float)
    n = A.shape[0]

    # Distance term: guard against zeros & negatives
    D = np.array(dist_mat, dtype=float)
    D = np.maximum(D, eps)         # avoid 0**negative
    np.fill_diagonal(D, np.inf)    # never consider self-edges
    with np.errstate(over='ignore'):
        cost_term = D ** eta       # finite because D>=eps

    while (A.sum() // 2) < target_m:
        # Recompute homophily each step
        k_mat = compute_homophily_matrix((A > 0).astype(int))
        k_safe = np.maximum(k_mat, eps)  # avoid 0**negative
        with np.errstate(over='ignore'):
            value_term = k_safe ** gamma

        # Candidate edges: only absent upper-triangular
        mask = np.triu((A == 0), k=1)
        prob = cost_term * value_term
        prob = prob * mask

        # Remove non-finite junk
        prob[~np.isfinite(prob)] = 0.0
        total = prob.sum()

        if total <= 0:
            # uniform fallback among missing edges
            cand = np.argwhere(mask)
            i, j = cand[rng.integers(0, len(cand))]
        else:
            cand = np.argwhere(mask)
            p_flat = prob[mask].ravel()
            p_flat = p_flat / p_flat.sum()  # renormalize
            i, j = cand[rng.choice(len(cand), p=p_flat)]

        A[i, j] = 1.0
        A[j, i] = 1.0

    return A


# ----------  Binarization for edges  ---------- # Needed for GNM fitting
def _binarize_for_edges(A, thr=0.0):
    A = np.asarray(A, float)
    A = (A + A.T) / 2.0
    np.fill_diagonal(A, 0.0)
    A_bin = (A > thr).astype(int)   # set your threshold here
    return A_bin

# ----------  Grid search + evaluation per subject  ---------- #

def fit_gnm_for_subject(A_obs, dist_mat, wide_eta=(-8, 0), wide_gamma=(-8, 8),
                        wide_grid=25, narrow_eta=(-3, 0), narrow_gamma=(0.1, 0.6),
                        narrow_grid=25, sample_probability=0.1, rng=None):
    """
    For one subject, perform a wide and narrow grid search to find the best (eta, gamma),
    optionally sampling some generated networks to run ESN analysis immediately.
    
    Returns:
      best_params: dict with gamma, eta, energy.
      sampled_models: list of (A_sim, eta, gamma) randomly chosen according to sample_probability.
      energy_grid: DataFrame with columns ['eta', 'gamma', 'energy'] from the narrow search.
    """
    rng = default_rng() if rng is None else rng

    n = A_obs.shape[0]
    # m_edges = int(A_obs.sum() // 2)
    
    A_bin = _binarize_for_edges(A_obs, thr=0.0)  # or pick a real threshold
    m_edges = int(np.triu(A_bin, k=1).sum())

    # Prepare seed network: 10% density network as in Mousley et al. (or use your own)
    # Here we use the union of edges present in >95% of all subjects; you can compute
    # this prior to looping over subjects. For simplicity we start with an empty seed.
    A_seed = np.zeros_like(A_obs)

    # Wide grid to estimate good region (coarse)
    wide_eta_values = np.linspace(wide_eta[0], wide_eta[1], wide_grid)
    wide_gamma_values = np.linspace(wide_gamma[0], wide_gamma[1], wide_grid)
    energies_wide = []
    for eta in wide_eta_values:
        for gamma in wide_gamma_values:
            A_sim = generate_gnm(A_seed, m_edges, dist_mat, gamma, eta, rng)
            energy = gnm_energy(A_obs, A_sim, dist_mat)
            energies_wide.append((eta, gamma, energy))
    # Identify roughly best region
    energies_wide = np.array(energies_wide, dtype=[('eta', float), ('gamma', float), ('energy', float)])
    best_eta_approx, best_gamma_approx = energies_wide['eta'][energies_wide['energy'].argmin()], \
                                         energies_wide['gamma'][energies_wide['energy'].argmin()]

    # Narrow grid around Mousley’s recommended range
    narrow_eta_values = np.linspace(narrow_eta[0], narrow_eta[1], narrow_grid)
    narrow_gamma_values = np.linspace(narrow_gamma[0], narrow_gamma[1], narrow_grid)
    energy_grid = []
    sampled_models = []
    best_energy = np.inf
    best_eta, best_gamma = None, None

    for eta in narrow_eta_values:
        for gamma in narrow_gamma_values:
            A_sim = generate_gnm(A_seed, m_edges, dist_mat, gamma, eta, rng)
            energy = gnm_energy(A_obs, A_sim, dist_mat)
            energy_grid.append({"eta": eta, "gamma": gamma, "energy": energy})
            # Keep track of best
            if energy < best_energy:
                best_energy = energy
                best_eta, best_gamma = eta, gamma
            # With some probability, save this network for later ESN analysis
            if rng.random() < sample_probability:
                sampled_models.append((A_sim.copy(), eta, gamma))

    energy_df = pd.DataFrame(energy_grid)
    best_params = {"eta": best_eta, "gamma": best_gamma, "energy": best_energy}
    return best_params, sampled_models, energy_df



# ----------  Full pipeline across subjects  ---------- #

def run_full_gnm_and_esn_pipeline(connectomes, dist_mat, sample_prob=0.1,
                                  n_sample_limit=100):
    """
    Run GNM fitting and ESN analysis. Performs grid search for each subject,
    collects best parameters and energy, then runs ESN analysis on a random
    subset of generated networks. Returns DataFrames with results.
    sample_prob controls how often to keep a generated GNM for ESN testing.
    """
    rng = default_rng(0)

    n_subj = connectomes.shape[2]
    best_param_rows = []
    energy_details = []
    sampled_networks = []

    # 1) Fit GNMs for all subjects and collect random samples
    for subj in range(n_subj):
        A_obs = connectomes[:, :, subj].copy()
        if A_obs.shape[0] != A_obs.shape[1]:
            raise ValueError(f"Connectome for subject {subj} is not square: {A_obs.shape}")
        best_params, samples, energy_grid = fit_gnm_for_subject(
            A_obs, dist_mat, wide_grid=3, narrow_grid=3, sample_probability=sample_prob, rng=rng) # wide_grid=10, narrow_eta=(-3, 0), narrow_gamma=(0.1, 0.6), narrow_grid=10, sample_probability=0.1, rng=None)
        
        best_param_rows.append({"network_index": subj,
                                "gamma": best_params["gamma"],
                                "eta": best_params["eta"],
                                "energy": best_params["energy"]})
        energy_grid["network_index"] = subj
        energy_details.append(energy_grid)
        # Add samples to global list
        for A_sim, eta, gamma in samples:
            sampled_networks.append({"A_sim": A_sim, "eta": eta, "gamma": gamma, "subject": subj})
        print(f"[Subj {subj}] best eta={best_params['eta']:.3f}, gamma={best_params['gamma']:.3f}, energy={best_params['energy']:.3f}")

    # Create DataFrames
    df_best = pd.DataFrame(best_param_rows)
    df_energy_details = pd.concat(energy_details, ignore_index=True)

    # 2) Sample networks for ESN analysis (limit total to n_sample_limit)
    rng.shuffle(sampled_networks)
    sampled_networks = sampled_networks[:n_sample_limit]
    esn_records = []

    for i, item in enumerate(sampled_networks):
        A_sim = item["A_sim"]
        subj_idx = item["subject"]
        eta = item["eta"]
        gamma = item["gamma"]
        # Build reservoir, evaluate memory capacity and sequence recall using your functions
        W = build_reservoir_from_connectome(A_sim, spectral_radius=0.99, rank=False)
        mc = evaluate_memory_capacity(W, n_lags=50, train_len=4000, test_len=1000,
                                      n_runs=10, random_state=i)
        seq = evaluate_sequence_recall(W, pattern_lengths=range(5, 26),
                                       train_trials=800, test_trials=200,
                                       n_runs=5, random_state=i)
        esn_records.append({
            "sample_index": i,
            "subject": subj_idx,
            "eta": eta,
            "gamma": gamma,
            "memory_capacity": mc,
            "sequence_recall": seq
        })
        # print(f"ESN sample {i}: subj={subj_idx}, eta={eta:.3f}, gamma={gamma:.3f}, MC={mc:.3f}, SR={seq:.3f}")

    df_esn = pd.DataFrame(esn_records)

    return df_best, df_energy_details, df_esn

# Example usage:
# connectomes: np.array shape (68, 68, 70)
# dist_mat: your (68,68) distance matrix
# df_best, df_energy_grids, df_esn = run_full_gnm_and_esn_pipeline(all_connectomes, dist_mat)

# Different numbers of connectomes: 
# less_connectomes = np.array([all_connectomes[:,:,0]]).T
less_connectomes = all_connectomes[:, :, :5]  # Use first 5 connectomes for testing
df_best, df_energy_grids, df_esn = run_full_gnm_and_esn_pipeline(less_connectomes, dist_mat)
# df_best.to_csv("gnm_best_params.csv", index=False)
# df_esn.to_csv("gnm_esn_results.csv", index=False)

# 5 in 1 minute. 

In [ ]:
df_energy_grids.to_csv(OUTPUT_PATH + "/gnm_energy_grids.csv", index=False)
df_energy_grids

In [ ]:
df_esn.to_csv(OUTPUT_PATH + "/gnm_esn_results.csv", index=False)
df_esn 


In [ ]:
df_best.to_csv(OUTPUT_PATH + "/gnm_best_params.csv", index=False)
df_best

In [ ]:

# Plotting 

import numpy as np
import matplotlib.pyplot as plt
from numpy.random import default_rng

def compute_energy_landscape(connectomes, dist_mat,
                             eta_range=(-3.0, 0.0), gamma_range=(0.1, 0.6),
                             grid_size=(100, 100), seed=0):
    """
    Compute a group energy landscape over an (eta, gamma) grid.
    For each grid cell we simulate ONE network for each subject and take the
    median energy across subjects (fast + smooth enough). Increase reps if needed.

    Returns
    -------
    etas, gammas : 1D arrays
    energy_grid  : (len(gammas), len(etas)) median energy across subjects
    """
    rng = default_rng(seed)
    n = connectomes.shape[0]
    m_subj = connectomes.shape[2]

    etas   = np.linspace(eta_range[0], eta_range[1], grid_size[0])
    gammas = np.linspace(gamma_range[0], gamma_range[1], grid_size[1])

    # target number of undirected edges per subject (match observed density)
    # target_m = [int(connectomes[:, :, s].sum() // 2) for s in range(m_subj)]
    target_m = []
    for s in range(m_subj):
        A_bin = _binarize_for_edges(connectomes[:, :, s], thr=0.0)  # or pick a real threshold
        m_edges = int(np.triu(A_bin, k=1).sum())
        target_m.append(m_edges)

    print(f"Target edges per subject: {target_m}")

    # start from empty seed (you can replace with a consensus 10% seed if you have it)
    A_seed = np.zeros((n, n), dtype=float)

    energy_grid = np.zeros((len(gammas), len(etas)), dtype=float)

    for gi, gamma in enumerate(gammas):
        for ei, eta in enumerate(etas):
            energies = []
            for s in range(m_subj):
                A_sim = generate_gnm(A_seed, target_m[s], dist_mat, gamma, eta, rng)
                E = gnm_energy(connectomes[:, :, s], A_sim, dist_mat)
                energies.append(E)
                print(f"Subject {s}, eta={eta:.3f}, gamma={gamma:.3f}, energy={E:.3f}")
            # Use median for robustness to stochasticity
            energy_grid[gi, ei] = float(np.median(energies))
    return etas, gammas, energy_grid


def plot_energy_landscape(etas, gammas, energy_grid, df_best=None,
                          title="", # 10,000 simulations",
                          vmin=None, vmax=None, savepath=None, show=True):
    """
    Plot a Figure-4-style heatmap with optional white dots for best-fit models.
    """
    fig, ax = plt.subplots(figsize=(6.2, 5.2)) # , dpi=150)

    # Heatmap: black=low (good), yellow=high (bad) like Mousley; 'hot' does that.
    im = ax.imshow(energy_grid,
                   origin="lower",
                   extent=[etas.min(), etas.max(), gammas.min(), gammas.max()],
                   aspect="auto",
                   cmap="hot",
                   vmin=vmin, vmax=vmax,
                   interpolation="bilinear")
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Energy", rotation=270, labelpad=12)

    # Axis labels/range
    ax.set_xlabel("η")
    ax.set_ylabel("γ")
    ax.set_title(title)

    # Overlay best-fit points (one per subject)
    if df_best is not None and {"eta","gamma"}.issubset(df_best.columns):
        ax.scatter(df_best["eta"].values, df_best["gamma"].values,
                #    s=8, c="white", edgecolors="none", alpha=0.95, zorder=3)
                   s=16, c="blue", edgecolors="none", alpha=0.95, zorder=3)
    if savepath is not None:
        fig.savefig(savepath, bbox_inches="tight")
    if show:
        plt.show()
    return fig, ax


In [ ]:
# Create figure 4 from Alexas paper
# 100 x 100 grid = ~10k sims  ➜ similar to the paper’s narrow sweep
number_individual_gammas = 50
number_individual_etas = 50
etas, gammas, E = compute_energy_landscape(all_connectomes, dist_mat,
                                        eta_range=(-8, 0), gamma_range=(-8, 8),
                                        # grid_size=(100, 100), seed=42) 
                                        grid_size=(number_individual_etas, number_individual_gammas), seed=42) # etas first, then gammas

print(etas.shape, gammas.shape, E.shape)

# 7.516666666666667 h (i.e 451 minutes) for 50x50 grid 

In [ ]:
#  Plot + overlay the white dots = each subject’s best-fit (eta, gamma)

title_part = f"{number_individual_gammas}_{number_individual_etas}"
plot_energy_landscape(etas, gammas, E, df_best=df_best,
                    title=f"{number_individual_etas} * {number_individual_gammas} simulations", savepath=OUTPUT_PATH + f"/figure4_like_heatmap_{title_part}.png")

# save the energy landscape data
energy_landscape_df = pd.DataFrame(E, columns=etas, index=gammas)
energy_landscape_df.index.name = 'gamma'
energy_landscape_df.columns.name = 'eta'
energy_landscape_df.to_csv(OUTPUT_PATH + f"/energy_landscape_{title_part}.csv")

df_best